# Training a PI-UF3 model on the HEA5 dataset

This notebook trains and evaluates pseudo-interaction UF3 (PI-UF3) models from scratch. It downloads
the five-element refractory alloy dataset of Byggmästar, Nordlund and Djurabekova (V, Nb, Mo, Ta, W),
featurizes it with UF3, trains PI-UF3 by alternating least squares (ALS), trains the standard UF3 model
on the same features for comparison, and evaluates both on the published test set.

Part A uses the full dataset with two-body terms and reproduces the two-body sweep of the paper to
the printed precision.
Part B adds three-body terms on a random subset of the structures at a coarser three-body resolution,
so that it runs on a workstation. The closing section lists what changes for the production settings.

**Requirements.** Linux or macOS, Python 3.11, about 6 GB of disk under the working directory, and
internet access for the download cells. On 8 cores Part A takes about 12 minutes and Part B about
35 minutes. Part A needs a few GB of memory. In Part B the PI-UF3 fits also stay within a few GB,
whereas the standard UF3 three-body baseline, a dense 16,610-parameter least-squares problem, peaks
near 24 GB; set `RUN_UF3_3B = False` below to skip it on a smaller machine. Every cell that writes
files is idempotent, so the notebook can be interrupted and rerun.

**Environment.** Create an environment with Jupyter before opening this notebook, for example:

```bash
mamba create -n pi_uf3 -c conda-forge python=3.11 numpy=1.26 scipy pandas pytables numba ase matplotlib jupyter pytest git pip
conda activate pi_uf3
jupyter notebook pi_uf3_hea5_tutorial.ipynb
```

The first code cells clone the fork of UF3 that contains PI-UF3 and install it into that environment.

In [ ]:
import os
from pathlib import Path

N_JOBS = min(16, len(os.sched_getaffinity(0)) if hasattr(os, "sched_getaffinity") else os.cpu_count())
SEED = 0                                  # seed of the random initialization of the factors
LEARNING_WEIGHT = 0.0                     # 0.0 trains on forces only, as in the paper; 0.5 balances energies and forces
WORKDIR = Path("pi_uf3_work").resolve()   # everything the notebook writes goes here
UF3_REPO = os.environ.get("PI_UF3_REPO", "https://github.com/sunghjung3/uf3.git")
UF3_BRANCH = "alchemy-gram-path-and-torch-checkpoint-fix"
RUN_TESTS = True                          # run the PI-UF3 unit tests after the install (about one minute)

# Part A: two-body terms, full dataset
P2_LIST = [1, 2, 3, 4, 5]                 # number of two-body pseudo-interactions to sweep
MAX_ITER, CHECKPOINT = 50, 10             # ALS sweeps and checkpoint interval

# Part B: two- and three-body terms, demonstration scale
N_TRAIN_3B, N_TEST_3B = 600, 300          # random subsets of the training and test sets; None uses all structures
RESOLUTION_3B = [5, 5, 10]                # knot intervals of the three-body basis; the paper uses [10, 10, 20]
P3_LIST = [1, 3, 5, 10]                   # number of three-body pseudo-interactions to sweep, at p2 = 4
RUN_UF3_3B = True                         # standard UF3 baseline for Part B; needs about 24 GB of memory

WORKDIR.mkdir(exist_ok=True)
print(f"{N_JOBS} workers, work directory ./{os.path.relpath(WORKDIR)}")

## Install the UF3 fork

PI-UF3 lives in the `uf3.alchemy` module of a fork of UF3. The cell clones the branch named above
unless the notebook already sits inside a checkout, and installs the package in editable mode into the
running environment.

In [ ]:
import importlib, subprocess, sys


def find_checkout():
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / "uf3" / "alchemy" / "alchemy.py").exists() and (base / "setup.py").exists():
            return base
    return None


SRC = find_checkout()
if SRC is None:
    SRC = WORKDIR / "uf3"
    if not SRC.exists():
        WORKDIR.mkdir(parents=True, exist_ok=True)
        subprocess.run(["git", "clone", "--branch", UF3_BRANCH, "--single-branch", UF3_REPO, SRC.name],
                       cwd=WORKDIR, check=True)

try:
    import uf3.alchemy.alchemy  # noqa: F401
    import fasteners  # noqa: F401
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(SRC)], check=True)
    sys.path.insert(0, str(SRC))   # the editable install is picked up at the next interpreter start
    importlib.invalidate_caches()

COMMIT = subprocess.run(["git", "-C", str(SRC), "rev-parse", "--short", "HEAD"],
                        capture_output=True, text=True).stdout.strip()
print(f"uf3 fork at ./{os.path.relpath(SRC)}, commit {COMMIT}")

In [ ]:
import collections, contextlib, hashlib, json, pickle, time, urllib.request, warnings
from concurrent.futures import ProcessPoolExecutor

import ase.io
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.interpolate import BSpline

import uf3
from uf3.alchemy import alchemy
from uf3.data import composition, io
from uf3.regression import least_squares as ls
from uf3.representation import bspline, process
from uf3.util import json_io

print("uf3 imported from", "./" + os.path.relpath(Path(uf3.__file__).parent))
print("numpy", np.__version__, "| scipy", __import__("scipy").__version__, "| ase", ase.__version__)

In [ ]:
if RUN_TESTS:
    # On a CPU model without a stored reference, one test writes the reference file and reports a skip.
    result = subprocess.run([sys.executable, "-m", "pytest", str(SRC / "tests" / "test_alchemy.py"), "-q"],
                            capture_output=True, text=True)
    print(result.stdout.strip().splitlines()[-1])
    assert result.returncode == 0, result.stdout[-3000:]

## Data

The dataset is published on Fairdata under CC BY 4.0
(https://etsin.fairdata.fi/dataset/2498ae37-7325-4e51-aed2-473b12ebae69). The training database
`db_HEA_v2.xyz` holds 2,859 structures and the separate test database `testset_HEA_v2_all.xyz` holds
1,532. The cell asks the Etsin download service for a signed URL of each file, downloads it, and checks
the SHA-256 checksum against the Metax file record.

In [ ]:
DATASET = "2498ae37-7325-4e51-aed2-473b12ebae69"
REMOTE_DIR = "/Mo-Nb-Ta-V-W_v2_potentials_and_data/data/Mo-Nb-Ta-V-W"
FILES = {"train": "db_HEA_v2.xyz", "test": "testset_HEA_v2_all.xyz"}
DATA_DIR = WORKDIR / "data"
DATA_DIR.mkdir(exist_ok=True)


def signed_url(remote_path):
    request = urllib.request.Request("https://etsin.fairdata.fi/api/download/authorize",
                                     data=json.dumps({"cr_id": DATASET, "file": remote_path}).encode(),
                                     headers={"Content-Type": "application/json"})
    with urllib.request.urlopen(request, timeout=120) as response:
        return json.load(response)["url"]


def sha256(path):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for block in iter(lambda: handle.read(1 << 20), b""):
            digest.update(block)
    return digest.hexdigest()


with urllib.request.urlopen(f"https://metax.fairdata.fi/rest/v2/datasets/{DATASET}/files", timeout=120) as response:
    checksums = {record["file_path"]: record["checksum"]["value"] for record in json.load(response)}

xyz = {}
for split, name in FILES.items():
    local, remote = DATA_DIR / name, f"{REMOTE_DIR}/{name}"
    if not local.exists() or sha256(local) != checksums[remote]:
        urllib.request.urlretrieve(signed_url(remote), local)
    assert sha256(local) == checksums[remote], name
    xyz[split] = local
    print(f"{split}: {name}, {local.stat().st_size / 1e6:.1f} MB, checksum verified")

UF3 reads the trajectories into one DataFrame keyed by structure. The keys `train_<i>` and
`test_<i>` select the two sets throughout the notebook. The training database contains dimers and
short-range repulsion configurations that the test database does not; they matter when training and
test errors are compared.

In [ ]:
coordinator = io.DataCoordinator()
for split in ("train", "test"):
    coordinator.dataframe_from_trajectory(str(xyz[split]), prefix=split)
df_data = coordinator.consolidate()
keys = {split: [k for k in df_data.index if k.startswith(split + "_")] for split in ("train", "test")}

config_types = {split: collections.Counter(atoms.info["config_type"] for atoms in ase.io.iread(str(xyz[split])))
                for split in ("train", "test")}
for split in ("train", "test"):
    atoms = int(df_data.loc[keys[split], "size"].sum())
    print(f"{split}: {len(keys[split])} structures, {atoms} atoms, {3 * atoms} force components, "
          f"{len(config_types[split])} configuration types")
small_separation = ["dimer", "short_range", "hea_short_range"]
print("small-separation types in train / test:",
      {t: (config_types['train'][t], config_types['test'][t]) for t in small_separation})

## Part A: two-body PI-UF3 on the full dataset

### Basis

The two-body basis is the one used in the paper: cubic B-splines on 32 knot intervals between 0 and
6 Å per element pair, with the last three coefficients fixed at zero so that the potential and its
derivatives vanish at the cutoff. Five elements give 15 pairs and 15 × 32 + 5 = 485 UF3 parameters.

In [ ]:
ELEMENTS = ["V", "Nb", "Mo", "Ta", "W"]


def make_basis(degree, resolution_3b=None):
    system = composition.ChemicalSystem(element_list=ELEMENTS, degree=degree)
    r_min = {pair: 0.0 for pair in system.interactions_map[2]}
    r_max = {pair: 6.0 for pair in system.interactions_map[2]}
    resolution = {pair: 32 for pair in system.interactions_map[2]}
    if degree == 3:
        r_min.update({trio: [0.0, 0.0, 0.0] for trio in system.interactions_map[3]})
        r_max.update({trio: [5.0, 5.0, 10.0] for trio in system.interactions_map[3]})
        resolution.update({trio: list(resolution_3b) for trio in system.interactions_map[3]})
    config = bspline.BSplineBasis(system, r_min_map=r_min, r_max_map=r_max, resolution_map=resolution,
                                  leading_trim=0, trailing_trim=3)
    if degree == 3:
        # The factorization needs the same basis for every triplet, so the reduced basis that
        # standard UF3 uses when both neighbors share an element is switched off.
        for trio in config.interactions_map[3]:
            config.symmetry[trio] = 1
        config.update_basis_functions()
    return config


def n_uf3_params(config):
    return sum(config.get_interaction_partitions()[0].values()) - len(config.frozen_c)


cfg2 = make_basis(2)
pairs = cfg2.interactions_map[2]
pair_labels = ["-".join(pair) for pair in pairs]
print(pair_labels)
print("UF3 parameters, two-body model:", n_uf3_params(cfg2))

### Featurization

Featurization is the standard UF3 step and is shared by UF3 and PI-UF3. Each structure becomes one
energy row and 3N force rows of B-spline features. The sparse HDF5 option keeps the files small.

In [ ]:
import resource

FEAT_DIR = WORKDIR / "features"
FEAT_DIR.mkdir(exist_ok=True)


def peak_gb():
    # peak resident memory so far of this process and of its finished workers, in GB
    return max(resource.getrusage(resource.RUSAGE_SELF).ru_maxrss,
               resource.getrusage(resource.RUSAGE_CHILDREN).ru_maxrss) / 1e6


def featurize(config, subset, out):
    if out.exists():
        print(f"{out.name}: exists")
        return out
    featurizer = process.BasisFeaturizer(config)
    start = time.time()
    with ProcessPoolExecutor(max_workers=N_JOBS) as client:
        featurizer.batched_to_hdf(str(out), df_data.loc[subset], client, n_jobs=N_JOBS, batch_size=10,
                                  progress="none", table_template="features_{}", sparse_hdf5=True)
    print(f"{out.name}: {len(subset)} structures in {time.time() - start:.0f} s, peak memory {peak_gb():.1f} GB")
    return out


feat2 = {split: featurize(cfg2, keys[split], FEAT_DIR / f"{split}_2b.h5") for split in ("train", "test")}

### Preprocessing for ALS

ALS revisits the training data many times, so PI-UF3 first converts the features of the training set
into a compact per-table format and records the energy and force weights. With `LEARNING_WEIGHT = 0`
the energy weight is zero and the fit uses forces only; the one-body terms are then fixed by the ridge
penalty alone and carry no information.

In [ ]:
REG_2B = dict(ridge_1b=1e-10, ridge_2b=1e-10, curvature_2b=1e-8)


def preprocess(config, feat, subset, out_dir):
    out_dir.mkdir(parents=True, exist_ok=True)
    files = dict(preprocessed_file=str(out_dir / "preprocessed.h5"),
                 coverage_file=str(out_dir / "coverage.npz"),
                 metadata_file=str(out_dir / "metadata.npz"))
    if not Path(files["metadata_file"]).exists():
        model = alchemy.AlchemicalModel(config, {k: 1 for k in range(2, config.degree + 1)})
        with open(out_dir / "preprocess.log", "w") as log, contextlib.redirect_stdout(log):
            model.preprocess_for_training(str(feat), subset, weight=LEARNING_WEIGHT, sparse_hdf5=True,
                                          progress="none", **files)
    meta = np.load(files["metadata_file"])
    print(f"{out_dir.name}: {int(meta['n_e'])} energies, {int(meta['n_f'])} force components, "
          f"w_e = {float(meta['w_e']):.3g}, w_f = {float(meta['w_f']):.3g}")
    return files


pre2 = preprocess(cfg2, feat2["train"], keys["train"], WORKDIR / "pre_2b")

### Training PI-UF3

`AlchemicalModel(config, {2: p2})` factorizes the 32 × 15 coefficient matrix of the two-body block into
a 32 × p2 matrix of pseudo-interaction coefficients and a 15 × p2 matrix of pseudo-weights. ALS
alternates closed-form solves of the two factors. The regularization dictionary uses the same
ridge and curvature strengths as the paper. With `cache_tables=True` the preprocessed tables stay in
memory between sweeps instead of being re-read from disk; the per-sweep log goes to `train.log` in
each fit directory.

In [ ]:
def n_pi_params(model):
    return sum(v.size for v in model.coeff.values()) + sum(v.size for v in model.pseudo_weights.values())


def train_pi_uf3(config, n_pseudo, pre, out_dir, reg):
    out_dir.mkdir(parents=True, exist_ok=True)
    np.random.seed(SEED)
    model = alchemy.AlchemicalModel(config, n_pseudo)
    start = time.time()
    with open(out_dir / "train.log", "w") as log, contextlib.redirect_stdout(log):
        model.fit_from_file(**pre, C_regularizers=alchemy.get_C_regularizers(config, **reg),
                            max_iter=MAX_ITER, checkpoint=CHECKPOINT, progress="none",
                            sparse_tables=True, cache_tables=True,
                            checkpoint_dir=str(out_dir / "checkpoint"),
                            tracker_filename=str(out_dir / "tracker.npz"),
                            train_iter_filename=str(out_dir / ".train_iter"))
    print(f"{out_dir.name}: {n_pi_params(model)} parameters, {MAX_ITER} ALS sweeps in {time.time() - start:.0f} s, "
          f"peak memory {peak_gb():.1f} GB")
    return model


pi2 = {p2: train_pi_uf3(cfg2, {2: p2}, pre2, WORKDIR / "fits_2b" / f"p{p2}", REG_2B) for p2 in P2_LIST}

### Training the standard UF3 model

The baseline is the ordinary UF3 least-squares fit on the same features, with the same weights and
the same regularization strengths. UF3 solves it in one step.

In [ ]:
def train_uf3(config, feat, subset, reg, out_dir):
    out_dir.mkdir(parents=True, exist_ok=True)
    model = ls.WeightedLinearModel(config, **reg)
    start = time.time()
    with open(out_dir / "train.log", "w") as log, contextlib.redirect_stdout(log):
        model.fit_from_file(str(feat), subset, weight=LEARNING_WEIGHT, sparse_hdf5=True, progress="none")
    print(f"{out_dir.name}: {n_uf3_params(config)} parameters, direct solve in {time.time() - start:.0f} s, "
          f"peak memory {peak_gb():.1f} GB")
    return model


uf3_2b = train_uf3(cfg2, feat2["train"], keys["train"], REG_2B, WORKDIR / "fits_2b" / "uf3")

### Evaluation

Both model classes predict through the same UF3 machinery, because PI-UF3 expands its factors into a
full UF3 coefficient vector at the end of training. The metric is the root-mean-square error of the
force components, in eV/Å. Energies are not reported for a force-only fit.

In [ ]:
def force_rmse(model, feat, subset):
    _, _, tables, _ = io.analyze_hdf_tables(str(feat))
    with contextlib.redirect_stdout(open(os.devnull, "w")), warnings.catch_warnings():
        warnings.filterwarnings("ignore", message="Processing in serial")   # prediction runs in one process by design
        _, _, y_f, p_f = model.batched_predict(str(feat), keys=subset, table_names=tables, score=False,
                                               sparse_hdf5=True, progress="none")
    return ls.rmse_metric(y_f, p_f)


def results_table(models, config, feats, subsets):
    rows = []
    for name, (model, n_params) in models.items():
        rows.append(dict(model=name, parameters=n_params,
                         train=force_rmse(model, feats["train"], subsets["train"]),
                         test=force_rmse(model, feats["test"], subsets["test"])))
    return pd.DataFrame(rows).set_index("model")


models_2b = {"UF3": (uf3_2b, n_uf3_params(cfg2))}
models_2b.update({f"PI-UF3, p2 = {p2}": (m, n_pi_params(m)) for p2, m in pi2.items()})
table_2b = results_table(models_2b, cfg2, feat2, keys)
table_2b.round(4)

In [ ]:
fig, ax = plt.subplots(figsize=(5, 3.5))
sizes = [n_pi_params(m) / n_uf3_params(cfg2) for m in pi2.values()]
ax.plot(sizes, [table_2b.loc[f"PI-UF3, p2 = {p2}", "test"] for p2 in pi2], "o-", label="PI-UF3, test")
ax.plot(sizes, [table_2b.loc[f"PI-UF3, p2 = {p2}", "train"] for p2 in pi2], "s--", label="PI-UF3, train")
ax.axhline(table_2b.loc["UF3", "test"], color="k", label="UF3, test")
ax.axhline(table_2b.loc["UF3", "train"], color="k", ls="--", label="UF3, train")
for p2, x in zip(pi2, sizes):
    ax.annotate(f"p2 = {p2}", (x, table_2b.loc[f"PI-UF3, p2 = {p2}", "test"]), textcoords="offset points",
                xytext=(4, 4), fontsize=8)
ax.set_xlabel("parameters relative to UF3")
ax.set_ylabel("force RMSE (eV/Å)")
ax.legend(fontsize=8)
fig.tight_layout()

### Looking inside the factorization

The product of the two factors is the physical object; the factors themselves carry a gauge freedom.
The scale is fixed by training, which normalizes each pseudo-weight column to unit root-mean-square, and
the sign is fixed here by orienting every pseudo-interaction so that its largest coefficient, which
lies in the repulsive core, is positive. The first cell checks that the product reproduces the pair
potentials stored in the expanded model.

In [ ]:
def fix_gauge(C, W):
    sign = np.sign(C[np.abs(C).argmax(0), np.arange(C.shape[1])])
    return C * sign, W * sign


def spline_curve(config, coefficients_free, r):
    knots = config.knots_map[pairs[0]]
    coefficients = np.concatenate([coefficients_free, np.zeros(config.trailing_trim)])
    return BSpline(knots, coefficients, 3)(r)


model = pi2[4]
C, W = fix_gauge(model.coeff[2], model.pseudo_weights[2])          # C: 32 x 4, W: 15 x 4
expanded = ls.arrange_coefficients(model.coefficients, cfg2)
for j, pair in enumerate(pairs):
    assert np.allclose((C @ W.T)[:, j], expanded[pair][:C.shape[0]], atol=1e-12)
print("C W^T reproduces every pair potential of the expanded model")

r = np.linspace(1.5, 6.0, 400)
fig, axes = plt.subplots(1, 2, figsize=(10, 3.6), gridspec_kw={"width_ratios": [1, 1.4]})
for p in range(C.shape[1]):
    axes[0].plot(r, spline_curve(cfg2, C[:, p], r), label=f"pseudo-interaction {p + 1}")
axes[0].axhline(0, color="k", lw=0.5)
axes[0].set_ylim(-1.5, 3)
axes[0].set_xlabel("r (Å)")
axes[0].set_ylabel("energy (eV)")
axes[0].legend(fontsize=8)
im = axes[1].imshow(W.T, aspect="auto", cmap="RdBu_r", vmin=-np.abs(W).max(), vmax=np.abs(W).max())
axes[1].set_xticks(range(len(pairs)), pair_labels, rotation=90, fontsize=8)
axes[1].set_yticks(range(W.shape[1]), [f"{p + 1}" for p in range(W.shape[1])])
axes[1].set_ylabel("pseudo-interaction")
fig.colorbar(im, ax=axes[1], label="pseudo-weight")
fig.tight_layout()

At p2 = 1 the gauge freedom is a single scale and sign, so the pseudo-weights are unique up to that
factor. Their magnitudes for the five homonuclear pairs follow the experimental melting points.

In [ ]:
MELTING_K = {"V": 2183, "Nb": 2750, "Mo": 2896, "Ta": 3290, "W": 3695}
w1 = np.abs(pi2[1].pseudo_weights[2][:, 0])
homonuclear = {pair[0]: w1[j] for j, pair in enumerate(pairs) if pair[0] == pair[1]}
elements = sorted(homonuclear, key=MELTING_K.get)
x = np.array([MELTING_K[e] for e in elements])
y = np.array([homonuclear[e] for e in elements])
print("ordering by pseudo-weight magnitude:", " < ".join(sorted(homonuclear, key=homonuclear.get)))
print(f"Pearson r between |w| and melting point: {np.corrcoef(x, y)[0, 1]:.3f}")
fig, ax = plt.subplots(figsize=(4, 3))
ax.plot(x, y, "o")
for e, xi, yi in zip(elements, x, y):
    ax.annotate(e, (xi, yi), textcoords="offset points", xytext=(5, -3))
ax.set_xlabel("melting point (K)")
ax.set_ylabel("|pseudo-weight|, p2 = 1")
fig.tight_layout()

### Exporting a model

`to_json` writes the expanded coefficients in the ordinary UF3 format, so a trained PI-UF3 model is
used exactly like a UF3 model afterwards: load it with `WeightedLinearModel.from_json`, or convert it
for the UF3 LAMMPS plugin. Nothing in the evaluation depends on the factorization.

In [ ]:
model_file = WORKDIR / "pi_uf3_2b_p4.json"
pi2[4].to_json(str(model_file))
reloaded = ls.WeightedLinearModel.from_json(str(model_file))
assert np.allclose(reloaded.coefficients, pi2[4].coefficients)
print(f"{model_file.name}: {model_file.stat().st_size / 1e3:.0f} kB, reloads to the same coefficients")

## Part B: adding three-body terms

The three-body block of UF3 has one coefficient tensor per element triplet, 75 triplets for five
elements. PI-UF3 factorizes that block in the same way, with p3 pseudo-interactions. The paper trains
on all 2,859 structures with a [10, 10, 20] three-body basis (1,320 coefficients per triplet, 99,485
UF3 parameters), which needs tens of gigabytes of memory for the UF3 baseline. This part runs on a
random subset of the structures at the coarser default resolution, so that it completes on a
workstation; the code path is the same. Even at this scale the dense UF3 solve is the memory-hungry
step, which is one of the points of the factorization.

In [ ]:
rng = np.random.default_rng(SEED)


def random_subset(all_keys, n):
    if n is None or n >= len(all_keys):
        return list(all_keys)
    return sorted(rng.choice(all_keys, n, replace=False), key=lambda k: int(k.rsplit("_", 1)[1]))


keys3 = {"train": random_subset(keys["train"], N_TRAIN_3B), "test": random_subset(keys["test"], N_TEST_3B)}
cfg3 = make_basis(3, RESOLUTION_3B)
trios = cfg3.interactions_map[3]
n_basis_3b = cfg3.get_interaction_partitions()[0][trios[0]]
print(f"{len(trios)} triplets, {n_basis_3b} coefficients per triplet, {n_uf3_params(cfg3)} UF3 parameters")
print(f"subsets: {len(keys3['train'])} training and {len(keys3['test'])} test structures")

feat3 = {split: featurize(cfg3, keys3[split], FEAT_DIR / f"{split}_3b.h5") for split in ("train", "test")}

In [ ]:
REG_3B = dict(REG_2B, ridge_3b=1e-7, curvature_3b=1e-10)
pre3 = preprocess(cfg3, feat3["train"], keys3["train"], WORKDIR / "pre_3b")
pi3 = {p3: train_pi_uf3(cfg3, {2: 4, 3: p3}, pre3, WORKDIR / "fits_3b" / f"p4-{p3}", REG_3B) for p3 in P3_LIST}
uf3_3b = train_uf3(cfg3, feat3["train"], keys3["train"], REG_3B, WORKDIR / "fits_3b" / "uf3") if RUN_UF3_3B else None

In [ ]:
models_3b = {"UF3": (uf3_3b, n_uf3_params(cfg3))} if RUN_UF3_3B else {}
models_3b.update({f"PI-UF3, p2 = 4, p3 = {p3}": (m, n_pi_params(m)) for p3, m in pi3.items()})
table_3b = results_table(models_3b, cfg3, feat3, keys3)
table_3b.round(4)

A three-body PI-UF3 model exports the same way. Standard UF3 would rebuild the basis with its
reduced form for triplets whose two neighbors share an element, so the basis is rebuilt with that
reduction switched off before the coefficients are loaded.

In [ ]:
model_file = WORKDIR / "pi_uf3_3b.json"
pi3[P3_LIST[-1]].to_json(str(model_file))
dump = json_io.load_interaction_map(str(model_file))
config = bspline.BSplineBasis.from_dict(dump)
for trio in config.interactions_map[3]:
    config.symmetry[trio] = 1
config.update_basis_functions()
reloaded = ls.WeightedLinearModel(config, regularizer=0, data_coverage=dump.get("data_coverage"))
reloaded.load(solution=dump)
assert np.allclose(reloaded.coefficients, pi3[P3_LIST[-1]].coefficients)
print(f"{model_file.name}: {model_file.stat().st_size / 1e6:.1f} MB, reloads to the same coefficients")

## From this notebook to the production settings

The fits in the paper differ from Part B in scale only. They use every structure, the three-body
resolution `[10, 10, 20]`, and the same regularization and sweep count. At that scale the UF3
baseline is a 99,485 × 99,485 linear system, and the fork provides a second training path,
`accumulate_gram` followed by `fit_from_gram`, that precomputes the normal-equation blocks once and
lets every model on the same features, including the standard UF3 model, restarts from other seeds,
and the joint L-BFGS comparison, reuse them. The `campaign` scripts that accompany the paper use that
path. For a first model the file-based path shown here is enough.